In [1]:
from pathlib import Path
import pandas as pd
import sys
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
LABELS_PATH = DATA_DIR / "labels.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.classifier import load_labeled_data, train_and_save_model

labels_df = load_labeled_data(str(LABELS_PATH))
labels_df.head()

,candidate_id,page_stub,page_url,image_url,local_path,domain,file_name,width,height,format,...,file_size_bytes,area,aspect_ratio,is_downloaded_and_valid,label,split,is_tiny,is_suspicious_domain,has_ui_keyword,target
0,page_01_cand_000000,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/105674087,c:\Users\Masha\ML_Projects\photo_classificatio...,mc.yandex.ru,105674087,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,False,0
1,page_01_cand_000001,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/27509004,c:\Users\Masha\ML_Projects\photo_classificatio...,mc.yandex.ru,27509004,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,test,True,True,False,0
2,page_01_cand_000002,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/26649402,c:\Users\Masha\ML_Projects\photo_classificatio...,mc.yandex.ru,26649402,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,val,True,True,False,0
3,page_01_cand_000003,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://counter.rambler.ru/top100.cnt?pid=2635991,c:\Users\Masha\ML_Projects\photo_classificatio...,counter.rambler.ru,top100.cnt,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,True,0
4,page_01_cand_000004,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://counter.rambler.ru/top100.cnt?pid=7728281,c:\Users\Masha\ML_Projects\photo_classificatio...,counter.rambler.ru,top100.cnt,1.0,1.0,GIF,...,43,1.0,1.0,True,non_content,train,True,True,True,0


In [2]:
labels_df['label'].value_counts(), labels_df['split'].value_counts()

(label
 content        306
 non_content    219
 Name: count, dtype: int64,
 split
 train    315
 test     105
 val      105
 Name: count, dtype: int64)

In [3]:
report = train_and_save_model(
    labels_csv_path=str(LABELS_PATH),
    model_path='models/best_model.pkl',
    model_type='logreg',
)
report['threshold']

0.6

In [4]:
metrics_df = pd.DataFrame([
    {'split': 'train', **report['train_metrics']},
    {'split': 'val', **report['val_metrics']},
    {'split': 'test', **report['test_metrics']},
])
metrics_df[['split', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 'tn']]

,split,precision,recall,f1,tp,fp,fn,tn
0,train,0.881356,0.565217,0.688742,104,14,80,117
1,val,0.911765,0.508197,0.652632,31,3,30,41
2,test,0.882353,0.491803,0.631579,30,4,31,40


In [5]:
Path('results').mkdir(parents=True, exist_ok=True)
metrics_df.to_csv('results/metrics.csv', index=False)
metrics_df

,split,precision,recall,f1,accuracy,tp,fp,fn,tn,n,mean_proba,median_proba,proba_p90
0,train,0.881356,0.565217,0.688742,0.701587,104,14,80,117,315,NaN,NaN,NaN
1,val,0.911765,0.508197,0.652632,0.685714,31,3,30,41,105,0.509617,0.518271,0.756359
2,test,0.882353,0.491803,0.631579,0.666667,30,4,31,40,105,0.500436,0.518948,0.767734
